# 8. Evaluation

Per outlet, from the saved article outputs. The two approaches in the same two spaces:

| | levels | score |
|---|---|---|
| classifier | majority level | expected score (probabilities × mean training score of each level) |
| regressor | NewsGuard bounds · training quotas · article vote | mean score · score remapped to training quantiles |
| no text | most frequent level | mean training score |

95% intervals: bootstrap over outlets (fixed seed). Classifier vs regressor: paired, on the same outlets.

In [ ]:
MODELS = "TF-IDF,Qwen3-8B,Mistral-24B,ModernBERT"
B = 2000

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import metrics as ME
import models as MO

art, outlets = MO.load()

In [ ]:
VIEWS = {}                                         # (model, approach, view) -> (space, true, predicted)
for name in MODELS.split(","):
    pa = MO.predictions("classification", name)
    if pa is not None:
        T = ME.outlets_classifier(pa, outlets)
        VIEWS.update({(name, "classifier", v): x for v, x in ME.classifier_views(T, outlets).items()})
    pa = MO.predictions("regression", name)
    if pa is not None:
        T = ME.outlets_regressor(pa, outlets)
        VIEWS.update({(name, "regressor", v): x for v, x in ME.regressor_views(T, pa, art).items()})
VIEWS.update({("no text", "baseline", v): x for v, x in ME.baseline_views(outlets).items()})
rows = [{"model": m, "approach": a, "view": v, "space": s, **ME.METRICS[s](y, p)} for (m, a, v), (s, y, p) in VIEWS.items()]
all_metrics = pd.DataFrame(rows); all_metrics.to_csv(C.OUT / "all_metrics.csv", index=False)
print(f"{len(all_metrics)} views; models: {sorted({m for m, _, _ in VIEWS})}")

In [ ]:
all_metrics[all_metrics.space == "levels"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

In [ ]:
all_metrics[all_metrics.space == "score"].dropna(axis=1, how="all").set_index(["model", "approach", "view"]).round(3)

## Intervals

In [ ]:
KEYS = {"levels": ["F1 macro", "tolerant (±1 level)", "quadratic kappa", "Spearman", "reliable: accuracy"],
        "score": ["MAE", "Spearman", "within 10 points", "within 20 points", "reliable: accuracy"]}
rows = []
for (m, a, v), (s, y, p) in VIEWS.items():
    ci, value = ME.bootstrap(y, p, s, KEYS[s], B=B), ME.METRICS[s](y, p)
    rows += [{"model": m, "approach": a, "view": v, "space": s, "metric": c, "value": value[c], "low": ci[c][0],
              "high": ci[c][1]} for c in KEYS[s]]
intervals = pd.DataFrame(rows); intervals.to_csv(C.OUT / "intervals.csv", index=False)
print(len(intervals), "intervals")

## Classifier vs regressor (regressor − classifier, same outlets)

In [ ]:
PAIRS = {"levels": ("majority level", ["levels: NewsGuard bounds", "levels: training quotas", "levels: article vote"]),
         "score": ("expected score", ["mean score", "remapped score"])}
rows = []
for name in MODELS.split(","):
    for space, (cls_view, reg_views) in PAIRS.items():
        if (name, "classifier", cls_view) not in VIEWS or (name, "regressor", reg_views[0]) not in VIEWS:
            continue
        for reg_view in reg_views:
            for key in KEYS[space]:
                d, (lo, hi) = ME.bootstrap_difference(VIEWS[(name, "regressor", reg_view)], VIEWS[(name, "classifier", cls_view)], key, B=B)
                rows.append({"model": name, "space": space, "regressor view": reg_view, "metric": key,
                             "difference": d, "low": lo, "high": hi, "not chance": not (lo <= 0 <= hi)})
pairs = pd.DataFrame(rows); pairs.to_csv(C.OUT / "classifier_vs_regressor.csv", index=False)
pairs.round(3)

## Probability metrics of the classifiers

In [ ]:
pd.DataFrame({name: ME.probability_metrics(ME.outlets_classifier(MO.predictions("classification", name), outlets))
              for name in MODELS.split(",") if MO.predictions("classification", name) is not None}).T.round(3)